# 01_01 · IM 主力连续一分钟数据：首尾锚定的 log 双向复权

从本地金融期货数据库读取 IM 各月份合约，生成同级的 `im_main_continuous_1m.parquet`。每次从头运行重建当前完整数据快照。

- 前一交易日成交量决定次日主力；只向后月换仓，后月成交量超过现主力的 1.10 倍时切换。
- 换月 log 价差取换月前最近共同有效交易日的收盘价：$g_k=\log C_{\mathrm{new},k}-\log C_{\mathrm{old},k}$。主力选择和价差参考日均早于换月日。
- 令 $G_i$ 为截至第 $i$ 个交易分钟的累计换月 log 价差。以完整序列的首分钟和末分钟为两个价格锚，取交易分钟权重 $w_i=i/(n-1)$：
  $$a_i=-(G_i-G_0)+w_i(G_{n-1}-G_0),\qquad
  \log P_i^{\mathrm{adj}}=\log P_i+a_i.$$
- 首尾两个分钟的调整量均为零，所以首分钟 open 与末分钟 close 保持原值。四个 OHLC 使用同一个分钟调整量，保留原始 OHLC、累计 log 价差、分配权重、调整量和调整后的 log OHLC／价格 OHLC。
- 累计换月价差按**交易分钟**线性分配，休市不增加权重。在消除换月 log 价差的同时，每相邻交易分钟的 log 收益增加固定漂移 $(G_{n-1}-G_0)/(n-1)$，因此片段内部收益也会改变。
- 这是本实验的“同时锚定首尾”口径。它使用整个快照的累计换月价差，新增未来换月或延长尾端会重算历史。`price_adjustment_available_at` 对全序列记录快照末分钟结束时刻；不能把这组价格当作历史当时可得的机器学习输入。
- 这是本地构造口径，不是聚宽官方 IM9999；首个上市日只提供选择信号。

使用 latitude 内核，在本 Notebook 所在目录运行。


**阶段阅读导航**：[01_01 主力连续价格](01_01_im_main_continuous.ipynb) → [02_01 BPV季节因子](02_01_im_bpv_intraday_seasonality.ipynb)（[02_02 窗口比较](02_02_im_bpv_window_comparison.ipynb)）→ [03_01 采样时钟与价格](03_01_im_clock_resampling.ipynb)（[03_02 时钟诊断](03_02_im_clock_comparison_diagnostics.ipynb)）→ [04_01 波动观测量](04_01_im_volatility_observations.ipynb) → [04_02 每日拟合](04_02_im_volatility_model_fitting.ipynb) → [04_03 下一采样步预测](04_03_im_next_step_volatility_forecasting.ipynb)／[04_04 未来15分钟预测](04_04_im_15m_volatility_forecasting.ipynb)。括号中的比较与诊断本用于查看对应阶段结果，不是下一阶段的数据生成依赖；最后两本预测相互独立。


## 1. 读取数据

In [1]:
import pathlib
import sys

# 按项目标记定位根目录，再导入项目配置。
project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()  # 当前工作目录

for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        sys.path.insert(0, str(candidate_root / "02_Futures_Lakehouse"))
        break
else:
    raise RuntimeError("未找到项目根目录")

from config.settings import settings

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

from config.data_contracts import (
    FUTURES_CONTRACT_CALENDAR_SCHEMA,
    arrow_to_pandas,
    validate_arrow_table,
)
from a00_03_notebook_schema_browser import display_schema_metadata

# 路径与日历读取契约；folder 供后续 cell 保存分钟序列。
folder = current_path
database_path = (
    candidate_root / "01_project_collection/JQ_strategy/financial_futures_data"
    / "data/warehouse/financial_futures.duckdb"
)
calendar_table_name = FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"table_name"].decode("utf-8")
calendar_partition_columns = (
    FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"partition_columns"].decode("utf-8").split(",")
)
calenda_root = settings.futures_lake_root / "silver" / calendar_table_name
calendar_columns = ["contract_code", "list_date", "delist_date"]
contract_schema = pa.schema([
    FUTURES_CONTRACT_CALENDAR_SCHEMA.field(column) for column in calendar_columns
])
calendar_partitioning = ds.partitioning(
    pa.schema([
        FUTURES_CONTRACT_CALENDAR_SCHEMA.field(column)
        for column in calendar_partition_columns
    ]),
    flavor="hive",
)
display_schema_metadata([FUTURES_CONTRACT_CALENDAR_SCHEMA])

# 只读本地 IM 日线，用实际交易日期限定日历读取范围。
with duckdb.connect(str(database_path), read_only=True) as connection:
    connection.execute("SET TimeZone='Asia/Shanghai'")
    daily_df = connection.execute("""
        SELECT contract_code, trading_date, close, volume, open_interest
        FROM futures_daily
        WHERE contract_code LIKE 'IM%.CCFX'
        ORDER BY trading_date, contract_code
    """).fetchdf()

if daily_df.empty:
    raise ValueError("本地金融期货数据库中没有 IM 日线数据")
daily_df["trading_date"] = daily_df["trading_date"].dt.date
dates = sorted(daily_df["trading_date"].unique())

# 从正式 silver 日历提取合约存续期，并按合约合并到日线。
calendar_dataset = ds.dataset(
    calenda_root, format="parquet", partitioning=calendar_partitioning,
)
calendar_table = calendar_dataset.to_table(
    columns=calendar_columns,
    filter=(
        (ds.field("exchange_code") == "CCFX")
        & (ds.field("underlying_code") == "IM")
        & (ds.field("trading_date") >= dates[0])
        & (ds.field("trading_date") <= dates[-1])
    ),
)
contract_table = validate_arrow_table(calendar_table, contract_schema)
contract_df = arrow_to_pandas(contract_table, contract_schema).drop_duplicates()
for column in ["list_date", "delist_date"]:
    contract_df[column] = pd.to_datetime(contract_df[column]).dt.date

daily_df = daily_df.merge(contract_df, on="contract_code", validate="many_to_one")
daily_df.head()


英文表名,中文表名,字段数,主键,Hive 分区,Schema 版本
dim_futures_contract_calendar,期货合约 Session 日历维度表,21,"contract_code,trading_date,session_number","exchange_code,year,month",1.2.0


,contract_code,trading_date,close,volume,open_interest,list_date,delist_date
0,IM2208.CCFX,2022-07-22,6956.6,22153.0,17517.0,2022-07-22,2022-08-19
1,IM2209.CCFX,2022-07-22,6901.2,6710.0,5858.0,2022-07-22,2022-09-16
2,IM2212.CCFX,2022-07-22,6737.0,3921.0,3613.0,2022-07-22,2022-12-16
3,IM2303.CCFX,2022-07-22,6588.0,3703.0,3338.0,2022-07-22,2023-03-17
4,IM2208.CCFX,2022-07-25,6878.6,22883.0,18747.0,2022-07-22,2022-08-19


## 2. 选择每日主力，累计换月 log 价差

In [2]:
daily_by_date = dict(tuple(daily_df.groupby('trading_date')))
contract_end = contract_df.set_index('contract_code')['delist_date'].to_dict()
closes = daily_df.pivot(index='trading_date', columns='contract_code', values='close')
current_contract = None
cumulative_roll_log_gap = 0.0
rows = []

for signal_date, trading_date in zip(dates[:-1], dates[1:]):
    signal = daily_by_date[signal_date]
    candidates = signal.loc[(signal['list_date'] <= trading_date)
                            & (signal['delist_date'] >= trading_date)
                            & (signal['volume'] > 0)].sort_values(
        ['volume', 'open_interest', 'delist_date', 'contract_code'],
        ascending=[False, False, True, True])
    if current_contract is None:
        selected = candidates.iloc[0]
    else:
        current = candidates.loc[candidates['contract_code'] == current_contract]
        later = candidates.loc[candidates['delist_date'] > contract_end[current_contract]]
        if current.empty:
            selected = later.iloc[0]
        elif not later.empty and later.iloc[0]['volume'] > 1.10 * current.iloc[0]['volume']:
            selected = later.iloc[0]
        else:
            selected = current.iloc[0]

    contract = selected['contract_code']
    is_roll = current_contract is not None and contract != current_contract
    if is_roll:
        common = closes.loc[:signal_date, [current_contract, contract]].dropna()
        common = common.loc[(common > 0).all(axis=1)].iloc[-1]
        cumulative_roll_log_gap += np.log(common[contract]) - np.log(common[current_contract])
    rows.append((trading_date, signal_date, contract, is_roll, cumulative_roll_log_gap))
    current_contract = contract

mapping_df = pd.DataFrame(rows, columns=['trading_date', 'signal_trading_date',
                                       'contract_code', 'is_roll_day', 'cumulative_roll_log_gap'])
mapping_df.head()

,trading_date,signal_trading_date,contract_code,is_roll_day,cumulative_roll_log_gap
0,2022-07-25,2022-07-22,IM2208.CCFX,False,0.0
1,2022-07-26,2022-07-25,IM2208.CCFX,False,0.0
2,2022-07-27,2022-07-26,IM2208.CCFX,False,0.0
3,2022-07-28,2022-07-27,IM2208.CCFX,False,0.0
4,2022-07-29,2022-07-28,IM2208.CCFX,False,0.0


## 3. 生成分钟序列，分配 log 价差并锚定首尾

复权权重随实际交易分钟递增，同一分钟四个价格使用同一调整量。首尾锚定是对完整快照的事后处理；这里记录其可用时刻，供下游传播。

In [3]:
with duckdb.connect(str(database_path), read_only=True) as connection:
    connection.execute("SET TimeZone='Asia/Shanghai'")
    connection.register("main_mapping", mapping_df)
    minute_df = connection.execute("""
        SELECT f.contract_code, f.trading_date, f.session_number, f.bar_at,
               f.open, f.high, f.low, f.close, f.volume, f.money, f.open_interest,
               m.signal_trading_date, m.is_roll_day, m.cumulative_roll_log_gap
        FROM futures_minute f JOIN main_mapping m
          ON f.contract_code=m.contract_code AND f.trading_date=m.trading_date
        ORDER BY f.bar_at
    """).fetchdf()

prices = ["open", "high", "low", "close"]
raw_price_values = minute_df[prices].to_numpy(dtype=np.float64)
if minute_df.empty or not np.isfinite(raw_price_values).all() or (raw_price_values <= 0).any():
    raise ValueError("log 双向复权要求非空、有限且为正的 OHLC")

# 全段首尾两个分钟的调整量为 0；休市不占交易分钟权重。
minute_count = len(minute_df)
cumulative_roll_log_gap = minute_df["cumulative_roll_log_gap"].to_numpy(dtype=np.float64)
relative_roll_log_gap = cumulative_roll_log_gap - cumulative_roll_log_gap[0]
total_roll_log_gap = float(relative_roll_log_gap[-1])
minute_df["bidirectional_adjustment_weight"] = (
    np.linspace(0.0, 1.0, minute_count) if minute_count > 1 else np.zeros(1)
)
minute_df["bidirectional_log_adjustment"] = (
    -relative_roll_log_gap + minute_df["bidirectional_adjustment_weight"] * total_roll_log_gap
)
minute_df["price_adjustment_available_at"] = minute_df["bar_at"].iloc[-1]
price_multiplier = np.exp(minute_df["bidirectional_log_adjustment"])
for price_column in prices:
    minute_df["bidirectional_adjusted_log_" + price_column] = (
        np.log(minute_df[price_column]) + minute_df["bidirectional_log_adjustment"]
    )
    # 与 exp(log(P)+a) 等价；乘法让 a=0 的首尾价格精确保留来源值。
    minute_df["bidirectional_adjusted_" + price_column] = minute_df[price_column] * price_multiplier

# 局部实验契约；不修改 silver 或原始合约行情。
schema = pa.schema([
    pa.field("contract_code", pa.string(), False), pa.field("trading_date", pa.date32(), False),
    pa.field("session_number", pa.int8(), False), pa.field("bar_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    *[pa.field(column, pa.float64(), False) for column in prices + ["volume", "money", "open_interest"]],
    pa.field("signal_trading_date", pa.date32(), False), pa.field("is_roll_day", pa.bool_(), False),
    pa.field("cumulative_roll_log_gap", pa.float64(), False),
    pa.field("bidirectional_adjustment_weight", pa.float64(), False),
    pa.field("bidirectional_log_adjustment", pa.float64(), False),
    pa.field("price_adjustment_available_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    *[pa.field("bidirectional_adjusted_log_" + column, pa.float64(), False) for column in prices],
    *[pa.field("bidirectional_adjusted_" + column, pa.float64(), False) for column in prices],
], metadata={
    b"price_adjustment_method": b"dual endpoint log adjustment; linear trading-minute allocation",
    b"price_adjustment_formula": b"a_i=-(G_i-G_0)+i/(n-1)*(G_last-G_0)",
    b"price_adjustment_scope": b"whole input snapshot; retrospective; future rolls can change history",
    b"price_adjustment_first_anchor_at": minute_df["bar_at"].iloc[0].isoformat().encode(),
    b"price_adjustment_last_anchor_at": minute_df["bar_at"].iloc[-1].isoformat().encode(),
    b"total_roll_log_gap": repr(total_roll_log_gap).encode(),
})
minute_table = validate_arrow_table(
    pa.Table.from_pandas(minute_df[schema.names], schema=schema, preserve_index=False), schema
)

# 验证首尾锚、log 恒等式和换月价差的分配关系。
np.testing.assert_array_equal(
    minute_df.loc[[0, minute_count - 1], ["bidirectional_adjusted_" + column for column in prices]].to_numpy(),
    raw_price_values[[0, minute_count - 1]],
)
log_adjustment = minute_df["bidirectional_log_adjustment"].to_numpy()
np.testing.assert_allclose(
    np.diff(log_adjustment) + np.diff(relative_roll_log_gap),
    np.diff(minute_df["bidirectional_adjustment_weight"]) * total_roll_log_gap,
    rtol=0, atol=1e-14,
)
for price_column in prices:
    np.testing.assert_allclose(
        np.log(minute_df["bidirectional_adjusted_" + price_column]),
        minute_df["bidirectional_adjusted_log_" + price_column],
        rtol=0, atol=2e-14,
    )

# 手工可推导案例：G=[0,0,0.2,0.2]，分配后的调整=[0,1/15,-1/15,0]。
example_roll_log_gap = np.array([0.0, 0.0, 0.2, 0.2])
example_weight = np.linspace(0.0, 1.0, len(example_roll_log_gap))
example_adjustment = -example_roll_log_gap + example_weight * example_roll_log_gap[-1]
np.testing.assert_allclose(example_adjustment, [0.0, 1/15, -1/15, 0.0], atol=1e-15)
# 无换月时调整恒为零；仅改变未来换月价差会改变早期调整，显式验证其事后属性。
np.testing.assert_array_equal(-np.zeros(4) + example_weight * 0.0, np.zeros(4))
changed_example_gap = np.array([0.0, 0.0, 0.3, 0.3])
changed_example_adjustment = -changed_example_gap + example_weight * changed_example_gap[-1]
assert changed_example_adjustment[1] != example_adjustment[1]

output_path = folder / "im_main_continuous_1m.parquet"
temporary_path = output_path.with_suffix(".parquet.tmp")
pq.write_table(minute_table, temporary_path, compression="zstd")
reread_minute_table = pq.ParquetFile(temporary_path).read()
assert reread_minute_table.schema.equals(minute_table.schema, check_metadata=True)
assert reread_minute_table.equals(minute_table)
temporary_path.replace(output_path)
print(f"{minute_count:,} 条分钟数据，{minute_df.trading_date.nunique()} 个交易日")
print(f"首端 open={minute_df.open.iloc[0]}；尾端 close={minute_df.close.iloc[-1]}")
print(f"累计换月 log 价差={total_roll_log_gap:.12f}")
print("首尾锚定、log 价差分配、事后依赖和 Parquet 复读验证通过")
print(output_path)
minute_df.head()


238,800 条分钟数据，995 个交易日
首端 open=6974.2；尾端 close=7666.6
累计换月 log 价差=-0.379603907631
首尾锚定、log 价差分配、事后依赖和 Parquet 复读验证通过
E:\Latitude_Analytics_v2\01_project_collection\JQ_strategy\draft_experiments\im_main_continuous_1m.parquet


,contract_code,trading_date,session_number,bar_at,open,high,low,close,volume,money,open_interest,signal_trading_date,is_roll_day,cumulative_roll_log_gap,bidirectional_adjustment_weight,bidirectional_log_adjustment,price_adjustment_available_at,bidirectional_adjusted_log_open,bidirectional_adjusted_open,bidirectional_adjusted_log_high,bidirectional_adjusted_high,bidirectional_adjusted_log_low,bidirectional_adjusted_low,bidirectional_adjusted_log_close,bidirectional_adjusted_close
0,IM2208.CCFX,2022-07-25,1,2022-07-25 09:31:00+08:00,6974.2,6998.0,6964.4,6973.8,664.0,927022080.0,17333.0,2022-07-22,False,0.0,0.000000,-0.000000,2026-08-28 15:00:00+08:00,8.849973,6974.200000,8.853380,6998.000000,8.848567,6964.400000,8.849916,6973.800000
1,IM2208.CCFX,2022-07-25,1,2022-07-25 09:32:00+08:00,6974.0,6995.2,6969.6,6982.8,428.0,597671840.0,17268.0,2022-07-22,False,0.0,0.000004,-0.000002,2026-08-28 15:00:00+08:00,8.849943,6973.988914,8.852978,6995.188880,8.849312,6969.588921,8.851204,6982.788900
2,IM2208.CCFX,2022-07-25,1,2022-07-25 09:33:00+08:00,6982.4,6989.8,6981.6,6989.8,280.0,391195160.0,17213.0,2022-07-22,False,0.0,0.000008,-0.000003,2026-08-28 15:00:00+08:00,8.851145,6982.377801,8.852204,6989.777778,8.851030,6981.577804,8.852204,6989.777778
3,IM2208.CCFX,2022-07-25,1,2022-07-25 09:34:00+08:00,6989.6,6993.0,6982.6,6993.0,320.0,447271480.0,17123.0,2022-07-22,False,0.0,0.000013,-0.000005,2026-08-28 15:00:00+08:00,8.852174,6989.566667,8.852660,6992.966651,8.851172,6982.566701,8.852660,6992.966651
4,IM2208.CCFX,2022-07-25,1,2022-07-25 09:35:00+08:00,6993.6,7009.0,6991.8,7002.2,286.0,400457120.0,17020.0,2022-07-22,False,0.0,0.000017,-0.000006,2026-08-28 15:00:00+08:00,8.852744,6993.555531,8.854944,7008.955433,8.852487,6991.755542,8.853973,7002.155476
